In [ ]:
# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from pathlib import Path
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import nltk
import re
import pandas as pd
import numpy as np

In [88]:
# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

REDDIT_PATH = Path("../data/raw/reddit/wsb_gme_archive.csv")
NEWS_PATH = Path("../data/raw/news/news_gdelt_GME.csv")
STOCK_DIR_RAW = Path("../data/raw/stock")

TICKERS = ["GME", "SPY", "AMC", "BB"]

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [89]:
# ---------------------------------------------------------
# Load raw datasets
# ---------------------------------------------------------

reddit_raw = pd.read_csv(REDDIT_PATH)
news_raw = pd.read_csv(NEWS_PATH, parse_dates=["published_at"])

stock_raw = {
    ticker: pd.read_csv(
        STOCK_DIR_RAW / f"{ticker.lower()}_stock.csv",
        parse_dates=["Date"]
    )
    for ticker in TICKERS
}

print("Reddit shape:", reddit_raw.shape)
print("News shape:", news_raw.shape)

for ticker, df in stock_raw.items():
    print(f"Stock ({ticker}) shape:", df.shape)

Reddit shape: (2439, 7)
News shape: (2069, 10)
Stock (GME) shape: (125, 7)
Stock (SPY) shape: (125, 7)
Stock (AMC) shape: (125, 7)
Stock (BB) shape: (125, 7)


In [90]:
# ---------------------------------------------------------
# Inspect raw datasets
# ---------------------------------------------------------

print("\n=== Reddit ===")
print(reddit_raw.columns.tolist())
display(reddit_raw.head())

print("\n=== News ===")
print(news_raw.columns.tolist())
display(news_raw.head())

for ticker, df in stock_raw.items():
    print(f"\n=== Stock ({ticker}) ===")
    print(df.columns.tolist())
    display(df.head())


=== Reddit ===
['created_utc', 'id', 'num_comments', 'score', 'selftext', 'title', 'url']


,created_utc,id,num_comments,score,selftext,title,url
0,1606793922,k4csaa,936,2543,>[Oh and uh short burn of the century comin so...,The REAL Greatest Short Burn of the Century Pa...,https://www.reddit.com/r/wallstreetbets/commen...
1,1607010382,k5zmcd,303,2251,"You've been holding GME for weeks, having dump...",Exactly how the GME squeeze will go for you.,https://www.reddit.com/r/wallstreetbets/commen...
2,1607031631,k66w5h,327,785,"GME has a new logo, graphics and slogan on all...",GME rebranding in progress - they've just chan...,https://www.reddit.com/r/wallstreetbets/commen...
3,1607042510,k6aa8p,0,1,[deleted],(GME) Gamestop's new branding looks strangely ...,https://i.redd.it/wkmgr34vg2361.jpg
4,1607043622,k6akre,0,1,[deleted],GME - GameStop's new branding looks strangely ...,https://i.redd.it/f90srt4lj2361.jpg



=== News ===
['url', 'url_mobile', 'title', 'seendate', 'socialimage', 'domain', 'language', 'sourcecountry', 'published_at', 'ticker']


,url,url_mobile,title,seendate,socialimage,domain,language,sourcecountry,published_at,ticker
0,https://www.barrons.com/articles/gamestop-stoc...,https://www.barrons.com/amp/articles/gamestop-...,GameStop Stock Doubled Last Week But Challenge...,20210115T233000Z,https://images.barrons.com/im-285024/social,barrons.com,English,China,2021-01-15 23:30:00+00:00,GME
1,https://www.marketwatch.com/articles/gamestop-...,NaN,GameStop Stock Doubled Last Week . The Challen...,20210115T234500Z,https://images.barrons.com/im-285024/social,marketwatch.com,English,United States,2021-01-15 23:45:00+00:00,GME
2,http://www.idahoreporter.com/2021/gme-stock-fa...,NaN,GME stock faces the mother of all short squee...,20210119T194500Z,http://www.idahoreporter.com/wp-content/upload...,idahoreporter.com,English,United States,2021-01-19 19:45:00+00:00,GME
3,https://www.benzinga.com/analyst-ratings/analy...,https://amp.benzinga.com/amp/content/19261270,"Gamestop Corporation ( NYSE : GME ), Best Buy ...",20210121T204500Z,NaN,benzinga.com,English,United States,2021-01-21 20:45:00+00:00,GME
4,https://www.marketwatch.com/articles/short-squ...,NaN,Short Squeeze Sends GameStop Shares to 2007 Le...,20210122T210000Z,https://images.barrons.com/im-288895/social,marketwatch.com,English,United States,2021-01-22 21:00:00+00:00,GME



=== Stock (GME) ===
['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']


...

,Date,Adj Close,Close,High,Low,Open,Volume
0,2020-10-01,311.627319,337.040009,338.739990,335.010010,337.690002,88698700
1,2020-10-02,308.668549,333.839996,337.010010,331.190002,331.700012,89431100
2,2020-10-05,314.142365,339.760010,339.959991,336.010010,336.059998,45713100
3,2020-10-06,309.676422,334.929993,342.170013,334.380005,339.910004,90128900
4,2020-10-07,315.066833,340.760010,341.630005,338.089996,338.119995,56999600



=== Stock (AMC) ===
['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']


,Date,Adj Close,Close,High,Low,Open,Volume
0,2020-10-01,46.500000,46.500000,48.000000,46.299999,47.799999,322380
1,2020-10-02,46.500000,46.500000,46.599998,44.200001,44.799999,357600
2,2020-10-05,41.299999,41.299999,43.599998,40.500000,43.000000,946880
3,2020-10-06,40.599998,40.599998,42.700001,40.500000,42.599998,840420
4,2020-10-07,40.400002,40.400002,41.099998,39.400002,40.799999,691770



=== Stock (BB) ===
['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume']


,Date,Adj Close,Close,High,Low,Open,Volume
0,2020-10-01,4.58,4.58,4.64,4.52,4.59,4971200
1,2020-10-02,4.44,4.44,4.50,4.37,4.49,6539300
2,2020-10-05,4.50,4.50,4.51,4.43,4.46,3106300
3,2020-10-06,4.55,4.55,4.68,4.51,4.53,3966000
4,2020-10-07,4.57,4.57,4.64,4.52,4.60,2166900


In [91]:
# ---------------------------------------------------------
# Basic data quality checks
# ---------------------------------------------------------

def inspect_dataset(df, name, duplicate_subset=None):
    print(f"\n=== {name} ===")

    print("\nShape:")
    print(df.shape)

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nData types:")
    print(df.dtypes)

    if duplicate_subset is not None:
        print(f"\nDuplicate {duplicate_subset}:")
        print(df.duplicated(subset=duplicate_subset).sum())


inspect_dataset(reddit_raw, "Reddit", duplicate_subset="id")
inspect_dataset(news_raw, "News", duplicate_subset="url")

for ticker, df in stock_raw.items():
    inspect_dataset(df, f"Stock ({ticker})", duplicate_subset="Date")


=== Reddit ===

Shape:
(2439, 7)

Missing values:
created_utc       0
id                0
num_comments      0
score             0
selftext        510
title             0
url               0
dtype: int64

Data types:
created_utc     int64
id                str
num_comments    int64
score           int64
selftext          str
title             str
url               str
dtype: object

Duplicate id:
0

=== News ===

Shape:
(2069, 10)

Missing values:
url                0
url_mobile       659
title              0
seendate           0
socialimage      399
domain             0
language           0
sourcecountry     19
published_at       0
ticker             0
dtype: int64

Data types:
url                              str
url_mobile                       str
title                            str
seendate                         str
socialimage                      str
domain                           str
language                         str
sourcecountry                    str
published_at    

## 4.1 Reddit Preprocessing

In [92]:
# ---------------------------------------------------------
# Inspect Reddit text fields
# ---------------------------------------------------------

print("Missing selftext:")
print(reddit_raw["selftext"].isna().sum())

print("\n[deleted] selftext:")
print(
    reddit_raw["selftext"]
    .fillna("")
    .str.strip()
    .str.lower()
    .eq("[deleted]")
    .sum()
)

print("\n[removed] selftext:")
print(
    reddit_raw["selftext"]
    .fillna("")
    .str.strip()
    .str.lower()
    .eq("[removed]")
    .sum()
)

print("\n[deleted] titles:")
print(
    reddit_raw["title"]
    .fillna("")
    .str.strip()
    .str.lower()
    .eq("[deleted]")
    .sum()
)

print("\n[removed] titles:")
print(
    reddit_raw["title"]
    .fillna("")
    .str.strip()
    .str.lower()
    .eq("[removed]")
    .sum()
)

Missing selftext:
510

[deleted] selftext:
308

[removed] selftext:
629

[deleted] titles:
0

[removed] titles:
0


In [93]:
# ---------------------------------------------------------
# Preprocess Reddit text
# ---------------------------------------------------------

reddit = reddit_raw.copy()

# Convert Unix timestamp to UTC datetime
reddit["created_at"] = pd.to_datetime(
    reddit["created_utc"],
    unit="s",
    utc=True
)

# Create calendar date for daily aggregation
reddit["date"] = reddit["created_at"].dt.date

# Clean post body
reddit["selftext_clean"] = (
    reddit["selftext"]
    .fillna("")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Treat deleted/removed post bodies as unavailable text
reddit["selftext_clean"] = reddit["selftext_clean"].replace(
    {
        "[deleted]": "",
        "[removed]": ""
    }
)

# Normalize post titles
reddit["title_clean"] = (
    reddit["title"]
    .fillna("")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Combine title and available post body
reddit["text"] = (
    reddit["title_clean"]
    + " "
    + reddit["selftext_clean"]
).str.strip()

print("Reddit shape:", reddit.shape)

print("\nMissing combined text:")
print(reddit["text"].isna().sum())

print("\nEmpty combined text:")
print(reddit["text"].eq("").sum())

display(
    reddit[
        [
            "created_at",
            "date",
            "title",
            "selftext_clean",
            "text"
        ]
    ].head()
)

Reddit shape: (2439, 12)

Missing combined text:
0

Empty combined text:
0


,created_at,date,title,selftext_clean,text
0,2020-12-01 03:38:42+00:00,2020-12-01,The REAL Greatest Short Burn of the Century Pa...,>[Oh and uh short burn of the century comin so...,The REAL Greatest Short Burn of the Century Pa...
1,2020-12-03 15:46:22+00:00,2020-12-03,Exactly how the GME squeeze will go for you.,"You've been holding GME for weeks, having dump...",Exactly how the GME squeeze will go for you. Y...
2,2020-12-03 21:40:31+00:00,2020-12-03,GME rebranding in progress - they've just chan...,"GME has a new logo, graphics and slogan on all...",GME rebranding in progress - they've just chan...
3,2020-12-04 00:41:50+00:00,2020-12-04,(GME) Gamestop's new branding looks strangely ...,,(GME) Gamestop's new branding looks strangely ...
4,2020-12-04 01:00:22+00:00,2020-12-04,GME - GameStop's new branding looks strangely ...,,GME - GameStop's new branding looks strangely ...


In [94]:
# ---------------------------------------------------------
# Inspect duplicate Reddit text
# ---------------------------------------------------------

print("Total Reddit posts:")
print(len(reddit))

print("\nUnique titles:")
print(reddit["title"].nunique())

print("\nDuplicated title rows:")
print(reddit["title"].duplicated().sum())

print("\nUnique combined texts:")
print(reddit["text"].nunique())

print("\nDuplicated combined-text rows:")
print(reddit["text"].duplicated().sum())


# ---------------------------------------------------------
# Most frequently repeated titles
# ---------------------------------------------------------

title_counts = (
    reddit["title"]
    .value_counts()
    .reset_index()
)

title_counts.columns = ["title", "count"]

print("\nMost frequently repeated titles:")
display(title_counts.head(20))


# ---------------------------------------------------------
# Most frequently repeated combined texts
# ---------------------------------------------------------

text_counts = (
    reddit["text"]
    .value_counts()
    .reset_index()
)

text_counts.columns = ["text", "count"]

print("\nMost frequently repeated combined texts:")
display(text_counts.head(20))

Total Reddit posts:
2439

Unique titles:
2337

Duplicated title rows:
102

Unique combined texts:
2344

Duplicated combined-text rows:
95

Most frequently repeated titles:


,title,count
0,Wall Street Bets Traders Are About To Be Crush...,4
1,"Chamath Palihapitiya on GameStop Stock (GME), ...",4
2,When GME ever closes above $690 for one single...,3
3,Gamestop Battle Speech That Everyone Holding $...,3
4,How is this even legal? GameStop (GME) - Execu...,3
5,Billionaire CRIES on National TV Because POOR ...,3
6,"""'You've already won' — Cramer tells investors...",3
7,"HAGENS BERMAN, NATIONAL TRIAL ATTORNEYS, Invit...",3
8,Technical Analysis today: GME (GameStop),3
9,The Gamestop of Brazil $GME $IRBR3 [PLEASE HEL...,3



Most frequently repeated combined texts:


,text,count
0,Wall Street Bets Traders Are About To Be Crush...,4
1,"Chamath Palihapitiya on GameStop Stock (GME), ...",4
2,When GME ever closes above $690 for one single...,3
3,Gamestop Battle Speech That Everyone Holding $...,3
4,How is this even legal? GameStop (GME) - Execu...,3
5,Billionaire CRIES on National TV Because POOR ...,3
6,"""'You've already won' — Cramer tells investors...",3
7,"HAGENS BERMAN, NATIONAL TRIAL ATTORNEYS, Invit...",3
8,Technical Analysis today: GME (GameStop),3
9,The Gamestop of Brazil $GME $IRBR3 [PLEASE HEL...,3


In [95]:
# ---------------------------------------------------------
# Check Exact Duplicate Text Within the Same Date
# ---------------------------------------------------------

reddit_same_day_dup = reddit[
    reddit.duplicated(
        subset=["date", "text"],
        keep=False
    )
].sort_values(["date", "text"])

print("Reddit")
print("Same-day duplicate rows:", len(reddit_same_day_dup))
print(
    "Unique duplicated date-text pairs:",
    reddit_same_day_dup[["date", "text"]]
    .drop_duplicates()
    .shape[0]
)

Reddit
Same-day duplicate rows: 143
Unique duplicated date-text pairs: 65


In [96]:
# ---------------------------------------------------------
# Remove Exact Same-Day Duplicate Reddit Posts
# ---------------------------------------------------------

reddit_before = len(reddit)

reddit = (
    reddit
    .drop_duplicates(
        subset=["date", "text"],
        keep="first"
    )
    .copy()
)

reddit_after = len(reddit)

print("Reddit observations before:", reddit_before)
print("Reddit observations after: ", reddit_after)
print("Duplicates removed:        ", reddit_before - reddit_after)

Reddit observations before: 2439
Reddit observations after:  2361
Duplicates removed:         78


## 4.2 News Preprocessing

In [97]:
# ---------------------------------------------------------
# Preprocess News text
# ---------------------------------------------------------

news = news_raw.copy()

# Standardize publication date
news["published_at"] = pd.to_datetime(
    news["published_at"],
    errors="coerce",
    utc=True
)

# Create calendar date for daily aggregation
news["date"] = news["published_at"].dt.date

# Normalize news titles for duplicate detection
news["title_clean"] = (
    news["title"]
    .fillna("")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Use headline as the news text
news["text"] = news["title_clean"]

print("News shape:", news.shape)

print("\nMissing publication dates:")
print(news["published_at"].isna().sum())

print("\nMissing text:")
print(news["text"].isna().sum())

print("\nEmpty text:")
print(news["text"].eq("").sum())

print("\nDuplicate URLs:")
print(news["url"].duplicated().sum())

display(
    news[
        [
            "published_at",
            "date",
            "title",
            "domain",
            "text"
        ]
    ].head()
)

News shape: (2069, 13)

Missing publication dates:
0

Missing text:
0

Empty text:
0

Duplicate URLs:
0


,published_at,date,title,domain,text
0,2021-01-15 23:30:00+00:00,2021-01-15,GameStop Stock Doubled Last Week But Challenge...,barrons.com,GameStop Stock Doubled Last Week But Challenge...
1,2021-01-15 23:45:00+00:00,2021-01-15,GameStop Stock Doubled Last Week . The Challen...,marketwatch.com,GameStop Stock Doubled Last Week . The Challen...
2,2021-01-19 19:45:00+00:00,2021-01-19,GME stock faces the mother of all short squee...,idahoreporter.com,GME stock faces the mother of all short squeez...
3,2021-01-21 20:45:00+00:00,2021-01-21,"Gamestop Corporation ( NYSE : GME ), Best Buy ...",benzinga.com,"Gamestop Corporation ( NYSE : GME ), Best Buy ..."
4,2021-01-22 21:00:00+00:00,2021-01-22,Short Squeeze Sends GameStop Shares to 2007 Le...,marketwatch.com,Short Squeeze Sends GameStop Shares to 2007 Le...


In [98]:
# ---------------------------------------------------------
# Inspect duplicate news titles
# ---------------------------------------------------------

title_counts = (
    news["title_clean"]
    .value_counts()
    .reset_index()
)

title_counts.columns = ["title_clean", "count"]

print("Unique titles:")
print(news["title_clean"].nunique())

print("\nDuplicated title rows:")
print(news["title_clean"].duplicated().sum())

print("\nTitles appearing more than once:")
print((title_counts["count"] > 1).sum())

print("\nMost frequently repeated titles:")
display(title_counts.head(20))

Unique titles:
1324

Duplicated title rows:
745

Titles appearing more than once:
197

Most frequently repeated titles:


,title_clean,count
0,The Reddit GameStop Boom Proves Were in a Meme...,173
1,The GameStop Fiasco Proves Were in a Meme Stoc...,57
2,Gamestop : Power to the Market Players ( Part 2 ),48
3,"GameStop slides , silver spree stalls as retai...",24
4,GameStop stock surges again weeks after high -...,15
5,Famed GameStop bull Roaring Kitty is a Massach...,14
6,Latest Articles,13
7,Analysis : A tulip by another name ? Gamestonk...,12
8,Gamestonk or silver ? Reddit investors worry a...,9
9,"3 Stocks Not to Spend Your $1 , 400 on",9


In [99]:
# ---------------------------------------------------------
# Inspect duplicate news titles across dates
# ---------------------------------------------------------

news_title_dates = (
    news
    .groupby("title_clean")
    .agg(
        article_rows=("title_clean", "size"),
        unique_dates=("date", "nunique"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        unique_domains=("domain", "nunique")
    )
    .sort_values("article_rows", ascending=False)
    .reset_index()
)

print("Repeated titles appearing on only one date:")
print(
    (
        (news_title_dates["article_rows"] > 1)
        & (news_title_dates["unique_dates"] == 1)
    ).sum()
)

print("\nRepeated titles appearing across multiple dates:")
print(
    (
        (news_title_dates["article_rows"] > 1)
        & (news_title_dates["unique_dates"] > 1)
    ).sum()
)

print("\nMost frequently repeated titles:")
display(news_title_dates.head(20))

Repeated titles appearing on only one date:
142

Repeated titles appearing across multiple dates:
55

Most frequently repeated titles:


,title_clean,article_rows,unique_dates,first_date,last_date,unique_domains
0,The Reddit GameStop Boom Proves Were in a Meme...,173,4,2021-01-27,2021-01-30,1
1,The GameStop Fiasco Proves Were in a Meme Stoc...,57,1,2021-01-27,2021-01-27,1
2,Gamestop : Power to the Market Players ( Part 2 ),48,1,2021-01-26,2021-01-26,1
3,"GameStop slides , silver spree stalls as retai...",24,1,2021-02-02,2021-02-02,17
4,GameStop stock surges again weeks after high -...,15,1,2021-02-25,2021-02-25,14
5,Famed GameStop bull Roaring Kitty is a Massach...,14,2,2021-01-29,2021-01-30,6
6,Latest Articles,13,4,2021-01-29,2021-02-12,1
7,Analysis : A tulip by another name ? Gamestonk...,12,2,2021-01-30,2021-01-31,7
8,"Bourse . Quest - ce que laffaire GameStop , qu...",9,1,2021-02-25,2021-02-25,9
9,"3 Stocks Not to Spend Your $1 , 400 on",9,1,2021-02-14,2021-02-14,8


In [100]:
# ---------------------------------------------------------
# Deduplicate news headlines within each calendar date
# ---------------------------------------------------------

news_before = len(news)

news = (
    news
    .sort_values("published_at")
    .drop_duplicates(
        subset=["date", "title_clean"],
        keep="first"
    )
    .reset_index(drop=True)
)

news_after = len(news)

print("News rows before deduplication:")
print(news_before)

print("\nNews rows after deduplication:")
print(news_after)

print("\nRows removed:")
print(news_before - news_after)

print("\nRemaining duplicate date-title pairs:")
print(
    news.duplicated(
        subset=["date", "title_clean"]
    ).sum()
)

News rows before deduplication:
2069

News rows after deduplication:
1392

Rows removed:
677

Remaining duplicate date-title pairs:
0


In [101]:
# ---------------------------------------------------------
# Inspect language distribution after deduplication
# ---------------------------------------------------------

print("News language distribution:")
print(news["language"].value_counts(dropna=False))

News language distribution:
language
English    1334
Spanish      25
French       24
Russian       4
German        3
Czech         2
Name: count, dtype: int64


In [102]:
# ---------------------------------------------------------
# Keep English-language news for text analysis
# ---------------------------------------------------------

news_before_language_filter = len(news)

news = (
    news[
        news["language"].eq("English")
    ]
    .reset_index(drop=True)
)

print("News rows before language filter:")
print(news_before_language_filter)

print("\nNews rows after language filter:")
print(len(news))

print("\nRows removed:")
print(news_before_language_filter - len(news))

News rows before language filter:
1392

News rows after language filter:
1334

Rows removed:
58


## 4.3 Stock Preprocessing

In [103]:
# ---------------------------------------------------------
# Preprocess stock data
# ---------------------------------------------------------

stock_frames = {}

for ticker, df in stock_raw.items():

    s = df.copy()

    s = s.rename(
        columns={
            "Date": "date",
            "Adj Close": "adj_close",
            "Close": "close",
            "High": "high",
            "Low": "low",
            "Open": "open",
            "Volume": "volume"
        }
    )

    s["date"] = pd.to_datetime(s["date"]).dt.date

    s = s.sort_values("date").reset_index(drop=True)

    # Daily log return, computed on adjusted close
    s["log_return"] = np.log(s["adj_close"] / s["adj_close"].shift(1))

    stock_frames[ticker] = s

    print(f"\n=== {ticker} ===")
    print("Shape:", s.shape)
    print("Date range:", s["date"].min(), "→", s["date"].max())
    print("Missing values:", s.isna().sum().sum(), "(first-day return is expected to be NaN)")
    print("Duplicate dates:", s["date"].duplicated().sum())


# ---------------------------------------------------------
# Build wide-format tables (one column per ticker) —
# matches the original project's stock_prices / stock_returns /
# stock_volume structure used downstream in analysis
# ---------------------------------------------------------

prices = pd.DataFrame({
    ticker: frame.set_index("date")["adj_close"]
    for ticker, frame in stock_frames.items()
})

returns = pd.DataFrame({
    ticker: frame.set_index("date")["log_return"]
    for ticker, frame in stock_frames.items()
})

volume = pd.DataFrame({
    ticker: frame.set_index("date")["volume"]
    for ticker, frame in stock_frames.items()
})

prices.index.name = "date"
returns.index.name = "date"
volume.index.name = "date"

print("\nPrices shape:", prices.shape)
print("Returns shape:", returns.shape)
print("Volume shape:", volume.shape)

display(prices.head())
display(returns.head())

# Keep a single-ticker GME frame around for the rest of this
# notebook (coverage/validation cells below expect one "stock" df)
stock = stock_frames["GME"]


=== GME ===
Shape: (125, 8)
Date range: 2020-10-01 → 2021-03-31
Missing values: 1 (first-day return is expected to be NaN)
Duplicate dates: 0

=== SPY ===
Shape: (125, 8)
Date range: 2020-10-01 → 2021-03-31
Missing values: 1 (first-day return is expected to be NaN)
Duplicate dates: 0

=== AMC ===
Shape: (125, 8)
Date range: 2020-10-01 → 2021-03-31
Missing values: 1 (first-day return is expected to be NaN)
Duplicate dates: 0

=== BB ===
Shape: (125, 8)
Date range: 2020-10-01 → 2021-03-31
Missing values: 1 (first-day return is expected to be NaN)
Duplicate dates: 0

Prices shape: (125, 4)
Returns shape: (125, 4)
Volume shape: (125, 4)


,GME,SPY,AMC,BB
date,,,,
2020-10-01,2.4425,311.627319,46.500000,4.58
2020-10-02,2.3475,308.668549,46.500000,4.44
2020-10-05,2.3650,314.142365,41.299999,4.50
2020-10-06,2.2825,309.676422,40.599998,4.55
2020-10-07,2.3400,315.066833,40.400002,4.57


,GME,SPY,AMC,BB
date,,,,
2020-10-01,NaN,NaN,NaN,NaN
2020-10-02,-0.039671,-0.009540,0.000000,-0.031045
2020-10-05,0.007427,0.017578,-0.118590,0.013423
2020-10-06,-0.035507,-0.014318,-0.017094,0.011050
2020-10-07,0.024880,0.017257,-0.004938,0.004386


In [104]:
# ---------------------------------------------------------
# Compare date coverage across datasets
# ---------------------------------------------------------

coverage = pd.DataFrame({
    "dataset": ["Reddit", "News", "Stock"],
    "start_date": [
        reddit["date"].min(),
        news["date"].min(),
        stock["date"].min()
    ],
    "end_date": [
        reddit["date"].max(),
        news["date"].max(),
        stock["date"].max()
    ],
    "rows": [
        len(reddit),
        len(news),
        len(stock)
    ],
    "unique_dates": [
        reddit["date"].nunique(),
        news["date"].nunique(),
        stock["date"].nunique()
    ]
})

display(coverage)

,dataset,start_date,end_date,rows,unique_dates
0,Reddit,2020-12-01,2021-03-30,2361,115
1,News,2021-01-15,2021-03-31,1334,72
2,Stock,2020-10-01,2021-03-31,125,125


In [105]:
# ---------------------------------------------------------
# Final validation of processed datasets
# ---------------------------------------------------------

print("=== Reddit ===")
print("Shape:", reddit.shape)
print("Missing text:", reddit["text"].isna().sum())
print("Empty text:", reddit["text"].eq("").sum())
print("Duplicate IDs:", reddit["id"].duplicated().sum())
print("Date range:", reddit["date"].min(), "→", reddit["date"].max())

print("\n=== News ===")
print("Shape:", news.shape)
print("Missing text:", news["text"].isna().sum())
print("Empty text:", news["text"].eq("").sum())
print(
    "Duplicate date-title pairs:",
    news.duplicated(subset=["date", "title_clean"]).sum()
)
print("Date range:", news["date"].min(), "→", news["date"].max())

print("\n=== Stock ===")
print("Shape:", stock.shape)
print("Missing values:", stock.isna().sum().sum())
print("Duplicate dates:", stock["date"].duplicated().sum())
print("Date range:", stock["date"].min(), "→", stock["date"].max())

=== Reddit ===
Shape: (2361, 12)
Missing text: 0
Empty text: 0
Duplicate IDs: 0
Date range: 2020-12-01 → 2021-03-30

=== News ===
Shape: (1334, 13)
Missing text: 0
Empty text: 0
Duplicate date-title pairs: 0
Date range: 2021-01-15 → 2021-03-31

=== Stock ===
Shape: (125, 8)
Missing values: 1
Duplicate dates: 0
Date range: 2020-10-01 → 2021-03-31


In [106]:
# ---------------------------------------------------------
# Select analysis-ready columns
# ---------------------------------------------------------

reddit_processed = reddit[
    [
        "id",
        "created_at",
        "date",
        "title_clean",
        "selftext_clean",
        "text",
        "score"
    ]
].copy()

news_processed = news[
    [
        "published_at",
        "date",
        "text",
        "domain"
    ]
].copy()

prices_processed = prices.copy()
returns_processed = returns.copy()
volume_processed = volume.copy()

In [107]:
# ---------------------------------------------------------
# Save processed datasets
# ---------------------------------------------------------

reddit_processed.to_csv(
    PROCESSED_DIR / "reddit_gme_processed.csv",
    index=False
)

news_processed.to_csv(
    PROCESSED_DIR / "news_gme_processed.csv",
    index=False
)

prices_processed.to_csv(PROCESSED_DIR / "stock_prices.csv")
returns_processed.to_csv(PROCESSED_DIR / "stock_returns.csv")
volume_processed.to_csv(PROCESSED_DIR / "stock_volume.csv")

print("Processed datasets saved.")

Processed datasets saved.
